# Embedding baseline: few-shot and zero-shot classification

Each augmented transcript (text with its tags) is encoded as a single vector with `sentence-transformers/bert-base-nli-mean-tokens`, and the FGW distance is replaced by a distance between vectors. Ability and benevolence are evaluated separately.

* **Few-shot** ($m$ labeled examples per level): each level is represented by the mean embedding of its $m$ examples, and every transcript gets the level of the nearest mean. For each fold, the $m$ examples per level come from the other folds, and the transcripts of the fold are classified.
* **Zero-shot** (unsupervised): $k$-means with $K = 3$ on the embeddings. Each cluster is mapped to a level with the Hungarian algorithm (the one-to-one assignment that maximizes agreement with the reference levels).

The reference level is either the **instructed** level (given to the LLM) or the **perceived** level (median of the participants' ratings). The protocol (data, folds, examples, scores) is shared with the other methods in `lib/evaluation.py`.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sys.path.append(os.path.realpath("../lib"))
from evaluation import (LEVELS, agreement_table, evaluate, load_or_create_split, load_predictions,
                        load_ratings, load_transcripts, map_clusters, plot_confusion, summarize,
                        zero_shot_subset)

## Parameters

In [ ]:
DATA_DIR = "../data_with_text_tag"
SPLIT_DIR = "splits"            # folds and few-shot examples, shared by all methods
OUTPUT_DIR = "results"

DIMENSIONS = ["ability", "benevolence"]
REFERENCE = "instructed"        # "instructed" or "perceived"
# Individual ratings of the participants, needed for REFERENCE = "perceived": csv with one row
# per rating and columns id (<condition>#<row>), dimension, rater, rating (Low / Medium / High).
# The perceived level is the median of the ratings; only rated transcripts are evaluated.
RATINGS_FILE = None

M = 10                          # labeled examples per level (few-shot)
N_FOLDS = 5
SEED = 0
N_PER_LEVEL = None              # evaluate at most this many transcripts per level (None: all)
ZERO_SHOT_N_PER_LEVEL = 100     # transcripts per level clustered in the zero-shot setting (None: all)

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
METRIC = "cosine"               # "cosine" or "euclidean"
N_KMEANS_INIT = 10              # k-means restarts

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Data

All `full_*` conditions. Each transcript has an `id` = `<condition>#<row>`. Folds and few-shot examples are loaded from `splits/` (created by the first notebook run, whichever method it is), so every method uses the same ones. **Keep the data parameters identical across notebooks.**

In [ ]:
df = load_transcripts(DATA_DIR, DIMENSIONS, REFERENCE, RATINGS_FILE, N_PER_LEVEL, SEED)
splits = {dimension: load_or_create_split(df[df["dimension"] == dimension], dimension, REFERENCE,
                                          SPLIT_DIR, M, N_FOLDS, SEED)
          for dimension in DIMENSIONS}
df.groupby(["dimension", "level"]).size().unstack()

## Embeddings

Cached in `results/embeddings.npz`, so each transcript is encoded once.

In [ ]:
cache_path = os.path.join(OUTPUT_DIR, "embeddings.npz")
cache = {}
if os.path.exists(cache_path):
    stored = np.load(cache_path, allow_pickle=True)
    cache = dict(zip(stored["ids"], stored["X"]))

texts = df.drop_duplicates("id").set_index("id")["text"]
missing = [i for i in texts.index if i not in cache]
if missing:
    model = SentenceTransformer(EMBEDDING_MODEL)
    vectors = model.encode(texts[missing].tolist(), batch_size=64, show_progress_bar=True)
    cache.update(zip(missing, vectors))
    np.savez(cache_path, ids=np.array(list(cache), dtype=object), X=np.array(list(cache.values())))


def embed(ids):
    X = np.array([cache[i] for i in ids])
    if METRIC == "cosine":  # cosine distance = Euclidean distance between normalized vectors
        X = X / np.linalg.norm(X, axis=1, keepdims=True)
    return X

## Few-shot: nearest mean embedding

In [ ]:
def few_shot(folds, examples):
    # Predicted level of every transcript, from the examples of its own fold.
    predictions = []
    for fold in range(N_FOLDS):
        test = folds[folds["fold"] == fold]
        support = examples[examples["fold"] == fold]
        centroids = np.array([embed(support.loc[support["level"] == level, "id"]).mean(axis=0)
                              for level in LEVELS])
        distances = np.linalg.norm(embed(test["id"])[:, None, :] - centroids[None, :, :], axis=2)
        predictions.append(pd.DataFrame({"id": test["id"], "fold": fold, "true": test["level"],
                                         "pred": np.array(LEVELS)[distances.argmin(axis=1)]}))
    return pd.concat(predictions, ignore_index=True)

## Zero-shot: $k$-means and Hungarian mapping

In [ ]:
def zero_shot(data):
    clusters = KMeans(n_clusters=len(LEVELS), n_init=N_KMEANS_INIT, random_state=SEED).fit_predict(embed(data["id"]))
    return pd.DataFrame({"id": data["id"], "true": data["level"].to_numpy(), "cluster": clusters,
                         "pred": map_clusters(clusters, data["level"])})

## Run

In [ ]:
predictions = {}
for dimension in DIMENSIONS:
    data = df[df["dimension"] == dimension]
    predictions[(dimension, "few-shot")] = few_shot(*splits[dimension])
    predictions[(dimension, "zero-shot")] = zero_shot(zero_shot_subset(data, ZERO_SHOT_N_PER_LEVEL, SEED))

## Results

Accuracy, macro-F1 and agreement with the reference: quadratic-weighted Cohen's $\kappa$ (levels are ordered, so predicting High for Low counts more than predicting Medium), with a 95% percentile bootstrap confidence interval over transcripts (1000 resamples).

Few-shot: mean ± standard deviation over folds. Chance level is 1/3 for accuracy and 0 for $\kappa$.

In [ ]:
results = evaluate(predictions, method="embeddings", reference=REFERENCE)
results.to_csv(os.path.join(OUTPUT_DIR, f"embedding_results_{REFERENCE}.csv"), index=False)
for (dimension, setting), pred in predictions.items():
    pred.to_csv(os.path.join(OUTPUT_DIR, f"embedding_predictions_{dimension}_{setting}_{REFERENCE}.csv"),
                index=False)
summarize(results)

All folds pooled; zero-shot also with ARI and AMI, which do not depend on the cluster-to-level mapping.

In [ ]:
pooled = results[results["fold"] == "all"].drop(columns="fold").round(3)
pooled.insert(pooled.columns.get_loc("kappa") + 1, "kappa [95% CI]",
              [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
               zip(pooled["kappa"], pooled["kappa_ci_low"], pooled["kappa_ci_high"])])
pooled

In [ ]:
plot_confusion(predictions, REFERENCE, os.path.join(OUTPUT_DIR, f"embedding_confusion_{REFERENCE}.png"))
plt.show()

## Agreement with human raters

AGREEMENT (quadratic-weighted Cohen's $\kappa$, 95% CI), only with `REFERENCE = "perceived"`.

* **Human raters vs human**: each rating compared with the median of the other raters of the same transcript (leave-one-rater-out). This is the agreement a single human reaches, i.e. the ceiling for the methods. Its CI resamples transcripts with all their ratings.
* **Method vs human**: the level predicted by the method compared with the perceived level (median of all raters). Its CI resamples transcripts.

In [ ]:
if REFERENCE == "perceived":
    ratings = load_ratings(RATINGS_FILE)
    methods = {"Embeddings": predictions}
    agreement = agreement_table(ratings, methods, DIMENSIONS)
    agreement.to_csv(os.path.join(OUTPUT_DIR, "agreement_embedding.csv"))
    display(agreement)
else:
    print("Set REFERENCE = \"perceived\" and RATINGS_FILE to compare with the human raters.")